# 08 · Đánh giá cuối trên tập test

Chấm 5 mô hình dự báo PM2.5 giờ kế tiếp trên cùng tập test theo thời gian. Persistence và Linear Regression là hai mốc đối chiếu. Random Forest, Gradient Boosting và MLP dùng các mô hình đã chốt từ notebook 05 và 07.

Đặt file trong `notebooks/` và dùng kernel đã chạy các notebook trước. Chọn **Run All** sau khi các mô hình đã được lưu. Với bản dữ liệu hiện tại, tập test dự kiến có **51.063 mẫu**. Nếu bật `SAVE_ARTIFACTS`, notebook xuất bảng điểm, biểu đồ và cập nhật nhật ký.

## 1. Nạp thư viện và xác định các mô hình

Dùng môi trường .venv cài theo requirements.txt (scikit-learn 1.7.2). Ưu tiên model trong models/; nếu baseline không có, tự dùng bản lưu trong outputs/experiments/experiments01/models/. Có thể đặt LINEAR_MODEL_PATH để chỉ định tệp khác. Bộ nạp kiểm tra phiên bản thư viện theo metadata trước khi nạp mô hình.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from threadpoolctl import threadpool_limits

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in (CURRENT_DIR, *CURRENT_DIR.parents) if (folder / "src").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Hãy mở notebook từ thư mục đồ án hoặc thư mục notebooks.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import TEST_START, load_data, time_split
from src.features import FFILL_LIMIT, PM25_LAGS, ROLLING_WINDOWS, add_features, split_xy

SAVE_ARTIFACTS = True
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs" / "notebooks"
TABLES_DIR = OUTPUTS_DIR / "tables"
EXPERIMENT_LOG = PROJECT_ROOT / "experiment_log.csv"

from src.artifacts import MODEL_LABELS, load_saved_model, resolve_model_paths
from src.evaluation import export_analysis

# None: prefer models/, fall back to the archived baseline from experiments01.
# A custom relative path is resolved against PROJECT_ROOT.
LINEAR_MODEL_PATH = None
MODEL_PATHS = resolve_model_paths(PROJECT_ROOT, linear_model_path=LINEAR_MODEL_PATH)
MODELS = {key: (MODEL_LABELS[key], path) for key, path in MODEL_PATHS.items()}
print("Resolved model paths:")
for key, (_, path) in MODELS.items():
    print(key, path.relative_to(PROJECT_ROOT) if path.is_relative_to(PROJECT_ROOT) else path)


## 2. Chuẩn bị tập test

Dùng các hàm trong `src/` giống notebook 07. `X_test` là bảng đặc trưng đưa cho mô hình, còn `y_test` chứa PM2.5 thực tế ở giờ kế tiếp để chấm điểm. Mốc chia dựa trên thời điểm của nhãn.

In [ ]:
featured = add_features(load_data())
labeled = featured.dropna(subset=["target"]).reset_index(drop=True)
train_validation, test = time_split(labeled)
test = test.reset_index(drop=True)

X_tv, y_tv = split_xy(train_validation)
X_test, y_test = split_xy(test)
assert train_validation["target_datetime"].max() < TEST_START <= test["target_datetime"].min()
assert X_tv.columns.equals(X_test.columns)
assert X_test.index.equals(y_test.index)

print(f"Train + validation: {len(X_tv):,} mẫu.")
print(f"Test: {len(X_test):,} mẫu, {X_test.shape[1]} đặc trưng.")
print(f"Thời gian nhãn test: {test['target_datetime'].min()} đến {test['target_datetime'].max()}.")

## 3. Dự báo trên cùng tập test

Persistence lấy PM2.5 ở giờ hiện tại làm dự báo giờ tiếp theo. Nếu giá trị hiện tại thiếu, dùng trung vị của `y_tv`, giống notebook 02. Với các mô hình còn lại, pipeline đã lưu tự áp dụng bước tiền xử lý khi gọi `predict`. Linear Regression dùng dự báo gốc như ở notebook 02.

In [ ]:
fallback = float(y_tv.median())
current_pm25 = X_test["PM2.5"]
predictions = {"baseline_persistence": current_pm25.fillna(fallback).to_numpy()}
labels = {"baseline_persistence": "Persistence"}
print(f"Persistence dùng giá trị dự phòng ở {current_pm25.isna().sum():,} mẫu.")

for key, (label, model_path) in MODELS.items():
    with threadpool_limits(limits=1):
        model = load_saved_model(key, model_path)
        predictions[key] = np.asarray(model.predict(X_test)).ravel()

    labels[key] = label
    print(f"{label}: nạp {model_path.name}, tạo {len(predictions[key]):,} dự báo.")

for key, values in predictions.items():
    if values.shape != (len(y_test),) or not np.isfinite(values).all():
        raise ValueError(f"Dự báo của {labels[key]} thiếu dòng hoặc có giá trị không hợp lệ.")

## 4. Tính MAE, RMSE và R²

MAE là độ lệch tuyệt đối trung bình. RMSE nhạy hơn với những dự đoán lệch nhiều. Cả hai đo bằng µg/m³, càng thấp càng tốt. R² càng cao càng tốt. Bảng xếp hạng theo RMSE test.

In [ ]:
scores = []
for key, prediction in predictions.items():
    scores.append({
        "experiment_id": f"{key}_test",
        "model": labels[key],
        "mae": mean_absolute_error(y_test, prediction),
        "rmse": np.sqrt(mean_squared_error(y_test, prediction)),
        "r2": r2_score(y_test, prediction),
        "n_test": len(y_test),
    })

test_metrics = pd.DataFrame(scores).sort_values("rmse", ignore_index=True)
test_metrics.insert(0, "rank", test_metrics["rmse"].rank(method="min").astype(int))
display(test_metrics[["rank", "model", "mae", "rmse", "r2", "n_test"]].round(4))

## 5. Biểu đồ và kết luận

Biểu đồ so sánh RMSE của 5 mô hình. Dùng kết quả để báo cáo hiệu năng trên test. Cấu hình các mô hình được chốt từ validation trước khi chấm.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(test_metrics["model"], test_metrics["rmse"], color="#39798e")
ax.invert_yaxis()
ax.set_xlabel("RMSE test (µg/m³)")
ax.set_title("So sánh các mô hình trên tập test")
ax.set_xlim(0, test_metrics["rmse"].max() * 1.16)
ax.bar_label(bars, fmt="%.2f", padding=4)
fig.tight_layout()
plt.show()

best = test_metrics.iloc[0]
baseline_rmse = test_metrics.loc[
    test_metrics["experiment_id"].eq("baseline_persistence_test"), "rmse"
].item()
print(f"RMSE test thấp nhất: {best['model']} ({best['rmse']:.4f} µg/m³).")
if baseline_rmse > 0:
    reduction = 100 * (baseline_rmse - best["rmse"]) / baseline_rmse
    print(f"RMSE thấp nhất giảm {reduction:.2f}% so với Persistence.")

## 6. Lưu kết quả và cập nhật nhật ký

Khi `SAVE_ARTIFACTS = True`, lưu bảng điểm vào `outputs/notebooks/tables/final_test_metrics.csv` và hình vào `outputs/notebooks/figures/final_test_rmse.png`. Nhật ký `experiment_log.csv` cập nhật theo mã thí nghiệm có đuôi `_test`. Hai dòng test cho Persistence và Linear Regression dùng cùng mã với notebook 02.

In [ ]:
if SAVE_ARTIFACTS:
    figures_dir = OUTPUTS_DIR / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)
    TABLES_DIR.mkdir(parents=True, exist_ok=True)
    metrics_path = TABLES_DIR / "final_test_metrics.csv"
    figure_path = figures_dir / "final_test_rmse.png"

    dataset_version = (
        f"beijing_air_quality_standardized.csv; ffill<={FFILL_LIMIT}h; "
        f"lags={'/'.join(map(str, PM25_LAGS))}; rolling={'/'.join(map(str, ROLLING_WINDOWS))}"
    )
    log_rows = (
        test_metrics[["experiment_id", "model", "mae", "rmse", "r2"]]
        .rename(columns={"model": "model_name"})
        .round({"mae": 6, "rmse": 6, "r2": 6})
    )
    log_rows.insert(2, "dataset_version", dataset_version)
    log_rows["notes"] = (
        f"Final test, {len(y_test)} rows with labels from {TEST_START:%Y-%m-%d}. "
        "Configurations fixed before test evaluation."
    )
    if EXPERIMENT_LOG.is_file() and EXPERIMENT_LOG.stat().st_size:
        existing_log = pd.read_csv(EXPERIMENT_LOG, encoding="utf-8-sig")
        existing_log = existing_log.loc[~existing_log["experiment_id"].isin(log_rows["experiment_id"])]
        log_rows = pd.concat([existing_log, log_rows], ignore_index=True)

    test_metrics.to_csv(metrics_path, index=False, encoding="utf-8-sig")
    fig.savefig(figure_path, dpi=150, bbox_inches="tight")
    log_rows.to_csv(EXPERIMENT_LOG, index=False, encoding="utf-8-sig")
    for path in (metrics_path, figure_path, EXPERIMENT_LOG):
        print("Đã lưu:", path.relative_to(PROJECT_ROOT))

## 7. Phân tích sai số và độ chắc chắn của mô hình đã chốt

Lưu riêng vào outputs/experiments/final_test/analysis/: dự đoán từng mẫu, sai số theo trạm/tháng/mùa/mức PM2.5, các trường hợp sai số lớn, hình residual và chuỗi thời gian, manifest truy vết, bảng so sánh RMSE train/validation/test (train_vs_validation_vs_test.csv; RMSE train chỉ là tham chiếu lạc quan vì mô hình đã fit trên các mẫu này) và bảng độ nhạy khi bỏ 0,1%/0,5%/1% mẫu sai lớn nhất (final_test_sensitivity.csv; quy tắc cố định trước, chỉ để báo cáo). Khoảng bootstrap ghép cặp dùng khối 7 ngày, giữ các trạm cùng thời điểm trong cùng khối. Khoảng này chỉ mô tả độ bất định có điều kiện trên các dự đoán cố định trong giai đoạn test đã quan sát; không dùng để chọn lại mô hình. Residual = y_true - y_pred; nhóm nồng độ đóng bên phải, ví dụ 35 thuộc <=35.


In [ ]:
prediction_frame = test[["station", "datetime", "target_datetime"]].reset_index(drop=True).copy()
prediction_frame["y_true"] = y_test.to_numpy()
for key, values in predictions.items():
    prediction_frame[f"pred_{key}"] = values

if SAVE_ARTIFACTS:
    analysis_tables = export_analysis(prediction_frame, MODEL_PATHS, train_validation=train_validation)
    display(analysis_tables["final_test_error_by_pm25_level.csv"])
    display(analysis_tables["paired_model_uncertainty.csv"])
    display(analysis_tables["train_vs_validation_vs_test.csv"].round(4))
    display(analysis_tables["final_test_sensitivity.csv"].round(4))
    print("Đã lưu phân tích bổ sung vào outputs/experiments/final_test/analysis/")
else:
    print("Bật SAVE_ARTIFACTS để xuất phân tích bổ sung.")


## 8. Kết quả validation dùng cho báo cáo

Ô sau chạy lại đúng các cấu hình hiện có: importance RF trên train/validation fold 5, ba seed MLP và kiểm tra epoch ở hai fold đầu. Không đổi model cuối hoặc chọn theo test. CSV và bảng xuất phát từ lần chạy notebook này; không nhập số bằng tay.


In [ ]:
from src.validation_diagnostics import run_diagnostics

if SAVE_ARTIFACTS:
    diagnostic_tables = run_diagnostics(return_tables=True)
    display(diagnostic_tables["rf_group_permutation_importance.csv"])
    display(diagnostic_tables["mlp_seed_summary.csv"])
    display(diagnostic_tables["mlp_validation_diagnostics.csv"])
else:
    diagnostic_tables = {}
    print("SAVE_ARTIFACTS=False: không chạy phần chẩn đoán có ghi file.")


## Bằng chứng notebook cho các bảng CSV

Các bảng dưới đây được xuất từ biến đã tính trong notebook, kèm mã ô và hash. scripts/verify_result_sources.py kiểm tra CSV khớp output số liệu của notebook; không dùng W&B/MLflow hoặc thêm thư viện.


In [ ]:
from src.notebook_evidence import record_tables, display_evidence
if SAVE_ARTIFACTS:
    from src.artifacts import load_saved_model
    config_rows=[]
    for key,path in MODEL_PATHS.items():
        fitted=load_saved_model(key,path)
        estimator=fitted.steps[-1][1]
        estimator=getattr(estimator,"regressor_",estimator)
        config_rows.append({
            "model": MODEL_LABELS[key], "estimator": type(estimator).__name__,
            "raw_input_columns": fitted.n_features_in_,
            "transformed_columns": len(fitted.steps[0][1].get_feature_names_out()),
            "layer_shapes": str([list(a.shape) for a in estimator.coefs_]) if hasattr(estimator,"coefs_") else None,
            "loss": estimator.get_params().get("loss"),
            "criterion": estimator.get_params().get("criterion"),
            "alpha": estimator.get_params().get("alpha")
        })
    model_configurations=pd.DataFrame(config_rows)
    improvement_rows=[]
    for reference in ["Persistence","Linear Regression"]:
        ref=test_metrics.loc[test_metrics.model.eq(reference)].iloc[0]
        for row in test_metrics.itertuples():
            improvement_rows.append({
                "model": row.model, "reference": reference,
                "rmse_reduction_percent": 100*(1-row.rmse/ref.rmse),
                "mae_reduction_percent": 100*(1-row.mae/ref.mae)
            })
    final_improvements=pd.DataFrame(improvement_rows)
    display(final_improvements)
    evidence_tables={"outputs/notebooks/tables/final_test_relative_improvements.csv":final_improvements,
                     "outputs/notebooks/tables/final_test_metrics.csv":test_metrics,
                     "outputs/notebooks/tables/final_model_configuration.csv":model_configurations,
                     **{f"outputs/experiments/final_test/analysis/{name}":table for name,table in analysis_tables.items()},
                     **{f"outputs/experiments/validation_diagnostics/{name}":table for name,table in diagnostic_tables.items()}}
    display(model_configurations)
    display_evidence(record_tables("notebooks/08_final_evaluation.ipynb","evidence-nb-08",evidence_tables))
